In [ ]:
import cv2
import numpy as np
import serial
import time

def detect_water_droplets_refined(image_path, serial_port, grid_size=20, min_edge=0.05, max_edge=0.3):
    img = cv2.imread(image_path)
    if img is None:
        print("이미지를 찾을 수 없습니다.")
        return
        
    img = cv2.resize(img, (640, 480))
    original_img = img.copy()
    
    height, width = img.shape[:2]
    cell_h = height // grid_size
    cell_w = width // grid_size
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    overlay = img.copy()
    
    detect_count = 0
    total_cells = grid_size * grid_size
    
    edges = cv2.Canny(gray, 50, 150)
    
    for r in range(grid_size):
        for c in range(grid_size):
            y1, y2 = r * cell_h, (r + 1) * cell_h
            x1, x2 = c * cell_w, (c + 1) * cell_w
            
            patch_edges = edges[y1:y2, x1:x2]
            edge_density = np.sum(patch_edges > 0) / (cell_w * cell_h)
            
            if min_edge < edge_density < max_edge:
                detect_count += 1
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 255), -1)
                
            cv2.rectangle(img, (x1, y1), (x2, y2), (200, 200, 200), 1)

    alpha = 0.4
    result_img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    combined_img = cv2.hconcat([original_img, result_img])
    
    # 20% 이상 감지 시 모터 가동
    motor_signal = 1 if (detect_count / total_cells) > 0.2 else 0

    print(f"총 {total_cells}개 구역 중 {detect_count}개 구역 물방울 감지")
    
    # 시리얼 통신 송신부
    if motor_signal == 1:
        print(">> 최종 모터 제어 신호: 가동 (1) -> MCU로 전송 완료")
        serial_port.write(b'1')  # 바이트 형태로 '1' 전송
    else:
        print(">> 최종 모터 제어 신호: 정상 (0) -> MCU로 전송 완료")
        serial_port.write(b'0')  # 바이트 형태로 '0' 전송
    
    cv2.imshow('Left: Original | Right: Detection Result', combined_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


# 실행 영역

PORT = 'COM5'  
BAUD_RATE = 9600

try:
    # 시리얼 통신 연결
    ser = serial.Serial(PORT, BAUD_RATE, timeout=1)
    time.sleep(2) # 보드 리셋 대기 시간 
    print(f"{PORT} 포트 시리얼 통신 연결 성공!")
    
    # 판별 및 신호 전송 실행
    detect_water_droplets_refined('rain1.jpg', serial_port=ser, grid_size=20)
    
    # 3. 통신 종료
    ser.close()

except serial.SerialException:
    print(f"오류: {PORT} 포트를 열 수 없습니다.")

COM5 포트 시리얼 통신 연결 성공!
총 400개 구역 중 294개 구역 물방울 감지
>> 최종 모터 제어 신호: 가동 (1) -> MCU로 전송 완료
